# **Business Recommendation & ROI**


In [ ]:
import sys
sys.path.append("..")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import joblib
import warnings
warnings.filterwarnings("ignore")

from src.data_loader import load_telco, split_features_target
from src.preprocessor import encode_features, add_engineered_features, split_data
from src.evaluate import find_optimal_threshold, compute_all_metrics
from src.shap_analysis import compute_roi_table

IMAGES = "../images"

df = load_telco("../data/Telco-Customer-Churn.csv")

X_raw, y = split_features_target(df)

X_enc   = encode_features(X_raw)
X_eng   = add_engineered_features(X_enc)
X_model = X_eng.select_dtypes(include=[np.number]).fillna(
    X_eng.select_dtypes(include=[np.number]).median()
)


In [ ]:
X_train, X_test, y_train, y_test = split_data(X_model, y)
best_model = joblib.load("../data/best_model.pkl")

y_prob = best_model.predict_proba(X_test)[:, 1]
optimal_thresh = find_optimal_threshold(y_test, y_prob, metric="f1")
y_pred = (y_prob >= optimal_thresh).astype(int)

## **1. ROI Analysis — Giá trị kinh tế của model**

In [ ]:
print("=== ROI Calculator — Chương trình Retention ===\n")
print("Assumptions (điều chỉnh theo thực tế của công ty):")
print("  Monthly revenue/customer: $65")
print("  Avg tenure lost khi churn: 12 tháng")
print("  Chi phí 1 retention offer: $50 (voucher, gọi điện)")
print("  Tỷ lệ thành công retention: 30%\n")

roi_df, net_benefit = compute_roi_table(
    y_test, y_prob,
    threshold=optimal_thresh,
    avg_monthly_revenue=65.0,
    avg_tenure_lost=12.0,
    retention_cost=50.0,
    retention_success_rate=0.30,
)
print(roi_df.to_string(index=False))
print(f"\n→ Net benefit của model: ${net_benefit:,.0f}")

In [ ]:
# Sensitivity analysis: ROI thay đổi theo retention_success_rate
success_rates = np.linspace(0.10, 0.60, 50)
net_benefits  = []

for rate in success_rates:
    _, nb = compute_roi_table(
        y_test, y_prob,
        threshold=optimal_thresh,
        retention_success_rate=rate,
    )
    net_benefits.append(nb)

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(success_rates, net_benefits, color="#378ADD", linewidth=2.5)
ax.fill_between(success_rates, 0, net_benefits,
                where=[nb > 0 for nb in net_benefits],
                alpha=0.15, color="#1D9E75", label="Profitable zone")
ax.fill_between(success_rates, 0, net_benefits,
                where=[nb <= 0 for nb in net_benefits],
                alpha=0.15, color="#D85A30", label="Loss zone")
ax.axhline(0, color="black", linewidth=1, linestyle="--")

breakeven = success_rates[np.argmin(np.abs(net_benefits))]
ax.axvline(breakeven, color="#D85A30", linewidth=2, linestyle=":",
           label=f"Break-even: {breakeven:.1%} success rate")

ax.set_xlabel("Retention Campaign Success Rate")
ax.set_ylabel("Net Benefit ($)")
ax.set_title("ROI Sensitivity Analysis\nNet benefit của model theo hiệu quả retention campaign",
             fontsize=12, fontweight="bold")
ax.xaxis.set_major_formatter(mticker.PercentFormatter(1.0))
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:,.0f}"))
ax.legend(fontsize=9)
ax.spines[["top", "right"]].set_visible(False)

plt.tight_layout()
plt.savefig(f"{IMAGES}/roi_sensitivity.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Break-even success rate: {breakeven:.1%}")
print("→ Nếu retention campaign thành công > {:.0%}, model có ROI dương".format(breakeven))

## **2. Customer Segmentation for Action**

In [ ]:
X_test_full = X_raw.iloc[X_test.index].copy() if hasattr(X_raw, "iloc") else X_test.copy()

results_df = X_test.copy()
results_df["churn_probability"] = y_prob
results_df["actual_churn"]      = y_test.values
results_df["predicted_churn"]   = y_pred
results_df["risk_tier"] = pd.cut(
    y_prob,
    bins=[0, 0.35, 0.65, 1.0],
    labels=["Low", "Medium", "High"],
)

# Segment statistics
seg_stats = results_df.groupby("risk_tier", observed=True).agg(
    n_customers=("churn_probability", "count"),
    avg_churn_prob=("churn_probability", "mean"),
    actual_churn_rate=("actual_churn", "mean"),
    avg_monthly=("MonthlyCharges", "mean") if "MonthlyCharges" in results_df.columns else ("churn_probability", "count"),
    avg_tenure=("tenure", "mean") if "tenure" in results_df.columns else ("churn_probability", "count"),
).reset_index()

print("=== Customer Segmentation by Risk ===")
print(seg_stats.round(2).to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
tiers   = ["Low", "Medium", "High"]
colors  = ["#1D9E75", "#EF9F27", "#D85A30"]

for ax, metric, title in zip(
    axes,
    ["n_customers", "actual_churn_rate", "avg_monthly"],
    ["Số Customers", "Actual Churn Rate", "Avg Monthly Charges ($)"],
):
    if metric not in seg_stats.columns:
        continue
    vals = [seg_stats[seg_stats["risk_tier"] == t][metric].values[0]
            if len(seg_stats[seg_stats["risk_tier"] == t]) > 0 else 0
            for t in tiers]
    bars = ax.bar(tiers, vals, color=colors, width=0.5)
    ax.set_title(title, fontsize=11, fontweight="bold")
    ax.spines[["top", "right"]].set_visible(False)
    for bar, val in zip(bars, vals):
        label = f"{val:.1%}" if "rate" in metric else f"{val:,.0f}"
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() * 1.02, label,
                ha="center", fontsize=10, fontweight="bold")

plt.suptitle("Customer Risk Segmentation — Action Targeting",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(f"{IMAGES}/customer_segmentation.png", dpi=150, bbox_inches="tight")
plt.show()

## **3. Business Recommendations — Final Report**

In [ ]:
high_risk = results_df[results_df["risk_tier"] == "High"]
med_risk  = results_df[results_df["risk_tier"] == "Medium"]

print("=" * 70)
print("BUSINESS RECOMMENDATIONS")
print("Telco Customer Churn Reduction Strategy")
print("=" * 70)

print(f"""
EXECUTIVE SUMMARY
─────────────────
Model: {type(best_model).__name__}
AUC-ROC: {compute_all_metrics(y_test, y_pred, y_prob)['AUC-ROC']:.3f}
Recall: {compute_all_metrics(y_test, y_pred, y_prob)['Recall']:.1%}
(Model phát hiện {compute_all_metrics(y_test, y_pred, y_prob)['Recall']:.1%} số khách sẽ churn)


SEGMENT 1 — HIGH RISK ({len(high_risk):,} customers, churn prob > 65%)
────────────────────────────────────────────────────────────────────────
Đặc điểm chung (từ SHAP analysis):
  • Month-to-month contract
  • Tenure < 12 tháng
  • Monthly charges cao
  • Không có TechSupport

ACTIONS (theo thứ tự ưu tiên):
  1. IMMEDIATE (trong 7 ngày):
     - Gọi điện retention call: offer discount 20% tháng tới
     - Propose upgrade sang One-year contract với giá lock-in

  2. SHORT-TERM (trong 30 ngày):
     - Free TechSupport upgrade 3 tháng
     - Loyalty point doubling cho tháng hiện tại

  3. LONG-TERM:
     - Enroll vào loyalty program tự động
     - Assign dedicated account manager nếu value cao


SEGMENT 2 — MEDIUM RISK ({len(med_risk):,} customers, churn prob 35-65%)
────────────────────────────────────────────────────────────────────────
ACTIONS:
  1. Email campaign: "Bạn đã gắn bó X tháng — cảm ơn bạn!" + small reward
  2. Survey satisfaction: tìm pain points trước khi họ quyết định rời
  3. Offer service upgrade với giá ưu đãi (cross-sell thêm dịch vụ)


SEGMENT 3 — LOW RISK (không cần action ngay)
────────────────────────────────────────────
  1. Standard loyalty program
  2. Upsell cơ hội — họ đang hài lòng, dễ mua thêm dịch vụ
  3. Referral program: khách gắn bó hay refer người khác


MODEL MONITORING
────────────────
□ Re-train model mỗi quý với data mới
□ Track precision/recall theo thời gian (model drift)
□ A/B test retention offers: đo conversion rate thực tế
□ Feed kết quả A/B test ngược lại model → cải thiện liên tục


EXPECTED IMPACT
───────────────
Nếu retention campaign đạt success rate 30%:
  Net benefit ước tính: ${net_benefit:,.0f}
  Break-even success rate: {breakeven:.1%}
  → Khả thi với một campaign retention tiêu chuẩn
""")

## **4. One-page Summary cho Slide**

In [ ]:
fig = plt.figure(figsize=(14, 8))
fig.patch.set_facecolor("white")

gs = fig.add_gridspec(2, 3, hspace=0.4, wspace=0.35)

# ── KPI boxes ──────────────────────────────────────────────────────────────
metrics_final = compute_all_metrics(y_test, y_pred, y_prob)
kpis = [
    ("AUC-ROC",  f"{metrics_final['AUC-ROC']:.3f}",  "#378ADD"),
    ("Recall",   f"{metrics_final['Recall']:.1%}",    "#1D9E75"),
    ("F1-Score", f"{metrics_final['F1']:.3f}",        "#7F77DD"),
    ("Net ROI",  f"${net_benefit:,.0f}",               "#D85A30"),
]

for i, (label, val, color) in enumerate(kpis):
    ax = fig.add_subplot(gs[0, i] if i < 3 else gs[1, i - 3])
    ax.set_facecolor(color + "18")
    ax.text(0.5, 0.65, val, ha="center", va="center",
            fontsize=22, fontweight="bold", color=color,
            transform=ax.transAxes)
    ax.text(0.5, 0.25, label, ha="center", va="center",
            fontsize=12, color="#444", transform=ax.transAxes)
    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_edgecolor(color)
        spine.set_linewidth(2)

# ── Risk distribution ─────────────────────────────────────────────────────
ax_risk = fig.add_subplot(gs[1, :2])
risk_counts_plot = results_df["risk_tier"].value_counts().sort_index()
tiers_order = [t for t in ["Low", "Medium", "High"] if t in risk_counts_plot.index]
vals_order  = [risk_counts_plot[t] for t in tiers_order]
colors_r    = [{"Low": "#1D9E75", "Medium": "#EF9F27", "High": "#D85A30"}[t] for t in tiers_order]
bars = ax_risk.bar(tiers_order, vals_order, color=colors_r, width=0.5)
ax_risk.set_title("Customer Risk Distribution", fontweight="bold")
ax_risk.spines[["top", "right"]].set_visible(False)
for bar, val in zip(bars, vals_order):
    ax_risk.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
                 str(val), ha="center", fontsize=11)

# ── Top features ──────────────────────────────────────────────────────────
ax_feat = fig.add_subplot(gs[1, 2])
if hasattr(best_model, "feature_importances_"):
    imp = pd.Series(best_model.feature_importances_, index=X_model.columns)
elif hasattr(best_model, "named_steps") and hasattr(best_model.named_steps.get("model", None), "feature_importances_"):
    imp = pd.Series(best_model.named_steps["model"].feature_importances_, index=X_model.columns)
else:
    imp = pd.Series(np.ones(len(X_model.columns)) / len(X_model.columns), index=X_model.columns)

top5 = imp.sort_values(ascending=True).tail(5)
ax_feat.barh(top5.index, top5.values, color="#378ADD", height=0.6)
ax_feat.set_title("Top 5 Features", fontweight="bold")
ax_feat.spines[["top", "right"]].set_visible(False)

fig.suptitle("Churn Prediction — Executive Dashboard",
             fontsize=16, fontweight="bold", y=1.01)
plt.savefig(f"{IMAGES}/executive_dashboard.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: executive_dashboard.png — Dùng làm slide đầu tiên khi present!")